# RRL CW1 — Problem 2 (MPC / iLQR)
This notebook is generated from `mpc.py` and is structured to produce the plots/values you need for **Problem 2 (a–c)**.

**Files used:** `mpc.py`, `common.py`, `my_cartpole_env.py`, `hyperparameters.py`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gymnasium.wrappers import TimeLimit

from my_cartpole_env import CartPoleEnv
from common import dynamics, quadratic_cost
from hyperparameters import *

plt.rcParams['figure.figsize'] = (8,4)

## Reuse the iLQR / MPC code (from `mpc.py`)
I’m copying the `CartpoleMPC` class as-is, so your results match the coursework code.

In [ ]:
class CartpoleMPC:
    def __init__(self, H=10, max_iters=5):

        # MPC/iLQR Parameters
        self.H = H          # Horizon length
        self.max_iters = max_iters    # iLQR iterations per time step
        ### FILL IN HERE ### hint: Q, R from provided cost function parameters
        self.Q = np.diag([1.0, 0.1, 10.0, 0.1])
        self.R = np.array([[0.01]])
        # raise NotImplementedError("MPC/iLQR Parameters not implemented")

        # Warm start buffer
        self.U_guess = np.zeros((self.H, 1))

    def get_jacobians(self, x, u):
        """Numerical Jacobians (Finite Difference) for A and B"""
        eps = 1e-6
        nx = len(x)
        nu = len(u)
        
        A = np.zeros((nx, nx))
        B = np.zeros((nx, nu))
        
        for i in range(nx):
            x_plus = x.copy()
            x_plus[i] += eps
            x_minus = x.copy()
            x_minus[i] -= eps
            A[:, i] = (dynamics(x_plus, u, continuous_action=True) - dynamics(x_minus, u, continuous_action=True)) / (2 * eps)
            
        for i in range(nu):
            u_plus = u.copy()
            u_plus[i] += eps
            u_minus = u.copy()
            u_minus[i] -= eps
            B[:, i] = (dynamics(x, u_plus, continuous_action=True) - dynamics(x, u_minus, continuous_action=True)) / (2 * eps)
            
        return A, B

    def solve_ilqr(self, x0, U_init):
        """The iLQR Solver"""
        U = U_init.copy()
        X = np.zeros((self.H + 1, 4))
        X[0] = x0
        
        # Initial Rollout
        for t in range(self.H):
            X[t+1] = dynamics(X[t], U[t], continuous_action=True)
            
        for _ in range(self.max_iters):
            # Backward Pass
            ks = [np.zeros((1, 1))] * self.H
            Ks = [np.zeros((1, 4))] * self.H
            
            # Terminal Value Function derivatives
            Vx = self.Q @ X[-1]
            Vxx = self.Q
            
            for t in reversed(range(self.H)):
                A, B = self.get_jacobians(X[t], U[t])

                # Gradients of the cost
                lx = self.Q @ X[t]
                lu = self.R @ U[t]

                ### FILL IN HERE ### hint: Q-function derivatives, control gains, value function update
                Qx  = lx + A.T @ Vx
                Qu  = lu + B.T @ Vx

                Qxx = self.Q + A.T @ Vxx @ A
                Quu = self.R + B.T @ Vxx @ B
                Qux = B.T @ Vxx @ A

                # Compute control gains
                Quu_inv = np.linalg.inv(Quu)
                ks[t] = -Quu_inv @ Qu
                Ks[t] = -Quu_inv @ Qux 

                # Update Value Function
                Vx = Qx + Ks[t].T @ Quu @ ks[t] + Ks[t].T @ Qu + Qux.T @ ks[t]
                Vxx = Qxx + Ks[t].T @ Quu @ Ks[t] + Ks[t].T @ Qux + Qux.T @ Ks[t]
                # raise NotImplementedError("iLQR not implemented")
                
            # Forward Pass (Line search simplified for brevity)
            X_new = np.zeros_like(X)
            X_new[0] = x0
            U_new = np.zeros_like(U)
            
            for t in range(self.H):
                ### FILL IN HERE ### hint: compute U_new[t] and X_new[t+1]
                U_new[t] = U[t] + ks[t] + Ks[t] @ (X_new[t] - X[t])
                X_new[t+1] = dynamics(X_new[t], U_new[t], continuous_action=True)
                # raise NotImplementedError("iLQR not implemented")
            
            X, U = X_new, U_new

        ks = np.array(ks)
        Ks = np.array(Ks)
        return U, X, ks, Ks

    def reset(self):
        """Reset the warm start buffer for a new episode"""
        self.U_guess = np.zeros((self.H, 1))
    
    def control(self, state):
        """MPC interface: solve and shift"""
        U_opt, _, _, _ = self.solve_ilqr(state, self.U_guess)
        
        # Extract first action and ensure it's a scalar
        action = U_opt[0, 0] if U_opt.ndim == 2 else U_opt[0]
        
        # Clip action to valid range [-1, 1]
        action = float(np.clip(action, -1.0, 1.0))
        
        # Warm start shift
        self.U_guess[:-1] = U_opt[1:]
        self.U_guess[-1] = 0
        
        return action

## Small helpers for evaluation
We’ll evaluate **cost** (same sign convention as DP/MPC in `common.py`) and also track whether an episode terminates early.

In [ ]:

def run_episode(env, policy, max_steps=MAX_EPISODE_STEPS, mode="MPC"):
    state, _ = env.reset()
    if mode == "MPC":
        policy.reset()
    total_cost = 0.0
    terminated_any = False

    for t in range(max_steps):
        if mode == "MPC":
            action = policy.control(state)
        else:
            raise ValueError("mode must be 'MPC'")
        state, _, terminated, truncated, _ = env.step(action)
        total_cost += quadratic_cost(state) + (TERMINAL_COST if terminated else 0.0)
        if terminated:
            terminated_any = True
            break
        if truncated:
            break
    return total_cost, terminated_any, t+1


def eval_policy(policy, H, disturbance=0.0, n_episodes=50, theta_max_deg=25.0):
    theta_thr = np.deg2rad(theta_max_deg)
    env = CartPoleEnv(x_threshold=X_LIMIT, theta_threshold_radians=theta_thr,
                      continuous_action=True, disturbance=disturbance)
    env = TimeLimit(env, max_episode_steps=MAX_EPISODE_STEPS)
    costs, term_flags, lengths = [], [], []
    for _ in range(n_episodes):
        c, term, L = run_episode(env, policy, mode="MPC")
        costs.append(c)
        term_flags.append(term)
        lengths.append(L)
    env.close()
    return {
        "mean_cost": float(np.mean(costs)),
        "std_cost": float(np.std(costs)),
        "terminated_rate": float(np.mean(term_flags)),
        "mean_len": float(np.mean(lengths)),
        "costs": np.array(costs),
        "term_flags": np.array(term_flags),
        "lengths": np.array(lengths),
    }


## 2(a) Derivatives: numerical vs analytic + structured A and B
### Numerical (finite differences)
- **Pros:** dead simple to implement, works even if dynamics are complicated / you don’t want to derive gradients.
- **Cons:** extra calls to dynamics (slow), sensitive to `eps`, can be noisy, and can be inaccurate near nonlinearities.

### Analytic (symbolic / hand-derived)
- **Pros:** fast and accurate, helps stability of backward pass (esp. `Quu`).
- **Cons:** derivations are error-prone and time-consuming, and changing the dynamics means re-deriving.

### Structured A and B (discrete-time)
Let state be **x = [x, ẋ, θ, θ̇]ᵀ** and discrete update (Euler) be:
- x⁺ = x + Δt ẋ
- ẋ⁺ = ẋ + Δt ẍ(x, ẋ, θ, θ̇, u)
- θ⁺ = θ + Δt θ̇
- θ̇⁺ = θ̇ + Δt θ̈(x, ẋ, θ, θ̇, u)

Then **A = ∂f/∂x** has the structure:
```
[ 1,  Δt,   0,    0 ]
[ 0,   1,  Δt∂ẍ/∂θ,  Δt∂ẍ/∂θ̇ ]   (plus ∂ẍ/∂x, ∂ẍ/∂ẋ terms if your ẍ depends on them)
[ 0,   0,   1,   Δt ]
[ 0,   0,  Δt∂θ̈/∂θ,  1+Δt∂θ̈/∂θ̇ ] (plus ∂θ̈/∂x, ∂θ̈/∂ẋ terms if present)
```
and **B = ∂f/∂u** has the structure:
```
[ 0 ]
[ Δt ∂ẍ/∂u ]
[ 0 ]
[ Δt ∂θ̈/∂u ]
```
Qualitatively: the “1” and “Δt” entries just reflect integration of velocities into positions/angles; the partial-derivative entries capture how force changes accelerations through the nonlinear coupling (sin/cos terms).

## 2(b) Horizon length sweep
We test **H ∈ [1,100]** in MPC mode and track:
- termination rate (should go to 0 when horizon is long enough)
- mean episode cost (you need the smallest H with mean cost < 0.01)

If this takes too long, reduce `N_EVAL` temporarily, then rerun around the boundary with a bigger value.

In [ ]:

THETA_MAX_DEG = 25.0
N_EVAL = 50      # change to 100+ for a cleaner estimate
MAX_H = 100

Hs = np.arange(1, MAX_H+1)
mean_costs = []
term_rates = []

for H in Hs:
    policy = CartpoleMPC(H=H, max_iters=5)
    out = eval_policy(policy, H, disturbance=0.0, n_episodes=N_EVAL, theta_max_deg=THETA_MAX_DEG)
    mean_costs.append(out["mean_cost"])
    term_rates.append(out["terminated_rate"])
    print(f"H={H:3d} | mean cost={out['mean_cost']:.4f} | term rate={out['terminated_rate']:.2f}", end="\r")

mean_costs = np.array(mean_costs)
term_rates = np.array(term_rates)

plt.figure()
plt.plot(Hs, term_rates)
plt.xlabel("Horizon H")
plt.ylabel("Termination rate")
plt.grid(True)
plt.show()

plt.figure()
plt.plot(Hs, mean_costs)
plt.xlabel("Horizon H")
plt.ylabel("Mean episode cost")
plt.grid(True)
plt.show()

# Extract answers requested
H_no_term = int(Hs[np.where(term_rates == 0.0)[0][0]]) if np.any(term_rates == 0.0) else None
H_min_cost = int(Hs[np.where(mean_costs < 0.01)[0][0]]) if np.any(mean_costs < 0.01) else None

H_no_term, H_min_cost


### Why very short horizon fails
With tiny H, the optimiser mostly prefers actions that reduce cost **immediately** (e.g., small θ right now), even if that choice sets you up for a worse θ̇ / ẋ a few steps later. The cartpole needs *planning* to cancel angular momentum; if you don’t look ahead far enough, you “win the next step” and lose the episode.

## 2(c) Disturbance recovery + iLQR comparison
### The small tweak for iLQR in the provided code
In the open-loop iLQR block, the computed `action` is a numpy array (shape like `(1,1)` or `(1,)`). `CartPoleEnv.step` expects a scalar in `[-1,1]` for continuous actions, so you should **convert to float and clip** before calling `step`.

Below, I implement iLQR rollouts in a clean function that does that.

### Disturbance sweep
We test increasing `DISTURBANCE` (extra force injected every 20 timesteps) and find the largest value where each method is still robust (low termination rate).

In [ ]:

def run_episode_ilqr(env, planner, H_ilqr=100):
    state, _ = env.reset()
    total_cost = 0.0

    # plan once
    u_plan, x_plan, ks, Ks = planner.solve_ilqr(state, np.zeros((H_ilqr, 1)))

    for t in range(H_ilqr):
        # local feedback around nominal
        u = u_plan[t] + ks[t] + Ks[t] @ (state - x_plan[t])
        action = float(np.clip(u.reshape(-1)[0], -1.0, 1.0))  # <-- the tweak
        state, _, terminated, truncated, _ = env.step(action)
        total_cost += quadratic_cost(state) + (TERMINAL_COST if terminated else 0.0)
        if terminated or truncated:
            return total_cost, terminated, t+1

    return total_cost, False, H_ilqr


def eval_ilqr(disturbance=0.0, n_episodes=50, theta_max_deg=25.0, H_ilqr=100):
    theta_thr = np.deg2rad(theta_max_deg)
    env = CartPoleEnv(x_threshold=X_LIMIT, theta_threshold_radians=theta_thr,
                      continuous_action=True, disturbance=disturbance)
    env = TimeLimit(env, max_episode_steps=MAX_EPISODE_STEPS)

    planner = CartpoleMPC(H=H_ilqr, max_iters=5)
    costs, terms = [], []
    for _ in range(n_episodes):
        c, term, _ = run_episode_ilqr(env, planner, H_ilqr=H_ilqr)
        costs.append(c)
        terms.append(term)
    env.close()
    return float(np.mean(costs)), float(np.mean(terms))


In [ ]:

# Use your Hmin from part (b) if you computed it.
# If H_min_cost came back None (because N_EVAL is small), pick something reasonable like 10-20.
Hmin = H_min_cost if H_min_cost is not None else 15
Hmin


In [ ]:

THETA_MAX_DEG = 25.0
N_EVAL = 50

dist_vals = np.linspace(0, 10, 21)  # sweep 0..10
mpc_term = []
ilqr_term = []

for d in dist_vals:
    mpc_policy = CartpoleMPC(H=Hmin, max_iters=5)
    out_mpc = eval_policy(mpc_policy, Hmin, disturbance=float(d), n_episodes=N_EVAL, theta_max_deg=THETA_MAX_DEG)
    mpc_term.append(out_mpc["terminated_rate"])

    _, term_ilqr = eval_ilqr(disturbance=float(d), n_episodes=N_EVAL, theta_max_deg=THETA_MAX_DEG, H_ilqr=100)
    ilqr_term.append(term_ilqr)

mpc_term = np.array(mpc_term)
ilqr_term = np.array(ilqr_term)

plt.figure()
plt.plot(dist_vals, mpc_term, label="MPC")
plt.plot(dist_vals, ilqr_term, label="iLQR (plan once)")
plt.xlabel("DISTURBANCE magnitude")
plt.ylabel("Termination rate")
plt.grid(True)
plt.legend()
plt.show()

# "Robustly withstand" threshold: you can pick 0% or <=10% termination
thr = 0.10
d_mpc = dist_vals[mpc_term <= thr].max() if np.any(mpc_term <= thr) else 0.0
d_ilqr = dist_vals[ilqr_term <= thr].max() if np.any(ilqr_term <= thr) else 0.0
d_mpc, d_ilqr


### Caregiving scenario answer (iLQR vs MPC)
If a person can stumble unpredictably, **MPC is the safer choice** because it re-plans from the *current measured state* each step. A one-shot iLQR plan (even with local feedback around the nominal trajectory) can degrade badly when disturbances push you into parts of the state space that the nominal linearisation never anticipated.